# Departure taxi-out time prediction: overview

The model predicts the **taxi-out time** of every departure: the time between leaving the stand (BLOCK) and take-off (MVT), in seconds.
- Training data: 12 months of 2025 at 10 European airports, departures and arrivals.
- Test data: departures in January, February, June and July 2026.

**Pipeline:**
1. **Features** (`02_features`): traffic, time, weather and group statistics from the raw movements, plus ADS-B tracks (adsb.lol) and OPDI ground events.
2. **Model A** (rows with a Network Manager off-block time AOBT_3, 98.6% of the test rows):
   - Target offset = BLOCK − AOBT_3. Prediction: taxi = max(60, (MVT − AOBT_3) − offset).
   - A five-component blend: XGBoost, two CatBoost models, Huber LightGBM and a locked-row mixture.
3. **Correctors:** an ADS-B track corrector, an OPDI ground-event corrector and an LTFM second stage learn the residual of the Model A taxi time.
4. **Model B** (rows without an NM off-block time): three airport-specific methods, then an ADS-B corrector and a residual corrector.
5. **Post-processing:** time shift, calibration to the published monthly taxi-out averages, and the LOBT window.

**Notebooks:**
- `01_external_data`: downloads external data (weather, METAR, ADS-B, OPDI).
- `02_features`: builds the feature tables.
- `03_training`: training. Uses the pretrained models by default; flags retrain each part.
- `04_prediction`: test prediction and the output file.
- `05_evaluation`: evaluation on held-out months of the training year.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
import config
import run
print("raw data folder:", config.RAW.name)

raw data folder: data


In [2]:
run.status()

raw data   data | training files 12 | test ['final_ranking.parquet', 'ranking.parquet']
ADS-B work/adsb | OPDI work/opdi
test months (1, 2, 6, 7) | template final_submitting.parquet | GPU False
missing inputs: none


departures: train 2,085,047 (with NM 2,062,577) | test 670,790 (with NM 661,713) | template 670,790
scales: ltfm_second=0.3, model_b_residual=1.0, monthly_calibration=1.0


## Data at a glance
Departure counts per month. The share with an NM off-block time is covered by Model A; the rest by Model B.

In [3]:
from taxi import data
e = data.departures("train", ["ADEP_mvt", "MVT_TIME_UTC_mvt", "AOBT_3_flt", "TAXITIME_SEC_mvt"])
t = data.departures("test", ["ADEP_mvt", "MVT_TIME_UTC_mvt", "AOBT_3_flt"])
summary = pd.concat([
    e.assign(split="train 2025", month=e.MVT_TIME_UTC_mvt.dt.month),
    t.assign(split="test 2026", month=t.MVT_TIME_UTC_mvt.dt.month)]).groupby(["split", "month"]).agg(
    departures=("ADEP_mvt", "size"), nm_share=("AOBT_3_flt", lambda s: s.notna().mean()))
summary.round(3)

departures  nm_share
split      month                      
test 2026  1          152719     0.984
           2          144158     0.991
           6          181791     0.986
           7          192122     0.985
train 2025 1          153706     0.991
           2          143732     0.992
           3          164449     0.993
           4          175288     0.994
           5          185202     0.991
           6          183142     0.986
           7          190713     0.979
           8          191182     0.988
           9          183950     0.987
           10         185674     0.990
           11         162332     0.992
           12         165677     0.990

In [4]:
e.groupby("ADEP_mvt")["TAXITIME_SEC_mvt"].describe(percentiles=[.1, .5, .9]).round(0)

,count,mean,std,min,10%,50%,90%,max
ADEP_mvt,,,,,,,,
EDDF,230141.0,863.0,330.0,2.0,478.0,837.0,1251.0,6907.0
EDDM,167334.0,812.0,302.0,2.0,533.0,774.0,1135.0,6899.0
EGLL,239546.0,1364.0,422.0,9.0,909.0,1319.0,1806.0,11452.0
EHAM,247951.0,785.0,318.0,-4.0,458.0,742.0,1157.0,11032.0
LEBL,179705.0,957.0,331.0,-8.0,595.0,906.0,1357.0,3604.0
LEMD,212242.0,1015.0,313.0,39.0,656.0,985.0,1402.0,3598.0
LFPG,239552.0,1020.0,453.0,-12.0,655.0,954.0,1444.0,84240.0
LIRF,160704.0,1195.0,1332.0,49.0,670.0,1025.0,1689.0,131167.0
LSZH,134907.0,741.0,385.0,-12.0,400.0,711.0,1080.0,87341.0


## Folders
| Folder | Contents |
|---|---|
| `taxi/` | code: `features/`, `model_a/`, `correctors/`, `model_b/`, `external/`, `postprocess.py`, `output.py` |
| `data/` | raw input files |
| `work/` | generated feature tables; `work/external/` downloaded weather, METAR and reference sources; `work/adsb/`, `work/opdi/` ADS-B and OPDI data |
| `artifacts/` | `oof/`: out-of-fold predictions of the training year (training base of the correctors); `reference/`: runway dimensions, stand–runway distances, stand centres |
| `models/` | trained models |
| `output/` | prediction files |